In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [2]:
# ── 1. CHARGER SPLITS ──────────────────────────
with open('splits_raw.pkl', 'rb') as f:
    splits_raw = pickle.load(f)

print("✅ Splits chargés")
for name, s in splits_raw.items():
    print(f"  {name} : Train={s['Xtr'].shape}")


✅ Splits chargés
  Type_Of_Failure : Train=(14658, 6)
  Serial_Device : Train=(14658, 7)
  Down_Time : Train=(14658, 9)
  When_Panne : Train=(14558, 13)


In [3]:
# ── 2. COLONNES À ENCODER ──────────────────────
CAT_COLS = [
    'Machine', 'Type Of Failure', 'Serial Device',
    'Defect Code', 'Shift', 'Equipe',
    'Last_Type_Failure', 'Last_Defect_Code',
    'Last_Serial_Device', 'Last_Shift'
]

In [4]:
# ── 3. LABEL ENCODING ──────────────────────────
label_encoders = {}
label_mappings = {}
splits_encoded = {}

for name, s in splits_raw.items():
    print(f"\n--- Encodage {name} ---")

    Xtr = s['Xtr'].copy()
    Xte = s['Xte'].copy()
    ytr = s['ytr'].copy()
    yte = s['yte'].copy()

    # Encoder chaque colonne catégorielle
    for col in CAT_COLS:
        if col not in Xtr.columns:
            continue

        le = LabelEncoder()

        # FIT sur train uniquement
        Xtr[col + '_enc'] = le.fit_transform(
            Xtr[col].fillna('INCONNU').astype(str))

        # TRANSFORM sur test
        Xte[col + '_enc'] = Xte[col].fillna(
            'INCONNU').astype(str).apply(
            lambda x: x if x in le.classes_
            else le.classes_[0])
        Xte[col + '_enc'] = le.transform(Xte[col + '_enc'])

        # Sauvegarder mapping
        mapping = dict(zip(
            le.classes_,
            le.transform(le.classes_)))
        label_encoders[f"{name}_{col}"] = le
        label_mappings[f"{name}_{col}"]  = mapping

        print(f"  ✅ {col}_enc "
              f"({len(le.classes_)} classes)")

        # Afficher mapping
        print(f"     Mapping (5 premiers) :")
        for k, v in list(mapping.items())[:5]:
            print(f"       '{k}' → {v}")

        # Supprimer colonne originale
        Xtr = Xtr.drop(columns=[col])
        Xte = Xte.drop(columns=[col])

    # Encoder target si classification
    task = ('reg' if name in ['Down_Time', 'When_Panne']
            else 'clf')

    if task == 'clf':
        le_target = LabelEncoder()
        ytr_enc   = le_target.fit_transform(
            ytr.fillna('INCONNU').astype(str))
        yte_enc   = yte.fillna('INCONNU').astype(str).apply(
            lambda x: x if x in le_target.classes_
            else le_target.classes_[0])
        yte_enc   = le_target.transform(yte_enc)

        label_encoders[f"{name}_target"] = le_target
        label_mappings[f"{name}_target"] = dict(zip(
            le_target.classes_,
            le_target.transform(le_target.classes_)))

        print(f"\n  ✅ Target encodé : "
              f"{len(le_target.classes_)} classes")
        print(f"     Mapping target :")
        for k, v in list(
                label_mappings[f"{name}_target"].items())[:5]:
            print(f"       '{k}' → {v}")

        splits_encoded[name] = {
            'Xtr'   : Xtr,
            'Xte'   : Xte,
            'ytr'   : ytr_enc,
            'yte'   : yte_enc,
            'feats' : list(Xtr.columns),
            'task'  : task,
        }
    else:
        splits_encoded[name] = {
            'Xtr'   : Xtr,
            'Xte'   : Xte,
            'ytr'   : ytr.values,
            'yte'   : yte.values,
            'feats' : list(Xtr.columns),
            'task'  : task,
        }




--- Encodage Type_Of_Failure ---
  ✅ Machine_enc (63 classes)
     Mapping (5 premiers) :
       'KOMAX_1' → 0
       'KOMAX_10' → 1
       'KOMAX_11' → 2
       'KOMAX_12' → 3
       'KOMAX_13' → 4
  ✅ Defect Code_enc (167 classes)
     Mapping (5 premiers) :
       'BC01' → 0
       'BC02' → 1
       'BC03' → 2
       'BC04' → 3
       'BC05' → 4
  ✅ Shift_enc (3 classes)
     Mapping (5 premiers) :
       'Après-midi' → 0
       'Matin' → 1
       'Nuit' → 2
  ✅ Equipe_enc (3 classes)
     Mapping (5 premiers) :
       'Equipe_1' → 0
       'Equipe_2' → 1
       'Equipe_3' → 2

  ✅ Target encodé : 18 classes
     Mapping target :
       'BLOC COUTEAUX' → 0
       'CONVOYEUR' → 1
       'DEMARRAGE PARC' → 2
       'ELECT/PNEUM' → 3
       'IT (SOFT/HARDWARE)' → 4

--- Encodage Serial_Device ---
  ✅ Machine_enc (63 classes)
     Mapping (5 premiers) :
       'KOMAX_1' → 0
       'KOMAX_10' → 1
       'KOMAX_11' → 2
       'KOMAX_12' → 3
       'KOMAX_13' → 4
  ✅ Type Of Failure_enc (

In [5]:
# ── 4. ONE HOT ENCODING (optionnel) ────────────
print("\n--- One Hot Encoding ---")
LOW_CARD = ['Shift', 'Equipe']

for name, s in splits_raw.items():
    Xtr = s['Xtr'].copy()
    Xte = s['Xte'].copy()

    cols_ohe = [c for c in LOW_CARD if c in Xtr.columns]
    if not cols_ohe:
        continue

    ohe = OneHotEncoder(
        sparse_output=False,
        handle_unknown='ignore')

    ohe_tr = ohe.fit_transform(Xtr[cols_ohe])
    ohe_te = ohe.transform(Xte[cols_ohe])

    ohe_cols  = ohe.get_feature_names_out(cols_ohe)
    df_ohe_tr = pd.DataFrame(
        ohe_tr, columns=ohe_cols,
        index=Xtr.index)
    df_ohe_te = pd.DataFrame(
        ohe_te, columns=ohe_cols,
        index=Xte.index)

    # Ajouter au dataset
    splits_encoded[name]['Xtr_ohe'] = pd.concat(
        [splits_encoded[name]['Xtr'], df_ohe_tr], axis=1)
    splits_encoded[name]['Xte_ohe'] = pd.concat(
        [splits_encoded[name]['Xte'], df_ohe_te], axis=1)

    print(f"  ✅ OHE {name} : {list(ohe_cols)}")




--- One Hot Encoding ---
  ✅ OHE Type_Of_Failure : ['Shift_Après-midi', 'Shift_Matin', 'Shift_Nuit', 'Equipe_Equipe_1', 'Equipe_Equipe_2', 'Equipe_Equipe_3']
  ✅ OHE Serial_Device : ['Shift_Après-midi', 'Shift_Matin', 'Shift_Nuit', 'Equipe_Equipe_1', 'Equipe_Equipe_2', 'Equipe_Equipe_3']
  ✅ OHE Down_Time : ['Shift_Après-midi', 'Shift_Matin', 'Shift_Nuit', 'Equipe_Equipe_1', 'Equipe_Equipe_2', 'Equipe_Equipe_3']
  ✅ OHE When_Panne : ['Equipe_Equipe_1', 'Equipe_Equipe_2', 'Equipe_Equipe_3']


In [6]:
# ── 5. SAUVEGARDER ─────────────────────────────
with open('splits_encoded.pkl', 'wb') as f:
    pickle.dump(splits_encoded, f)
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
with open('label_mappings.pkl', 'wb') as f:
    pickle.dump(label_mappings, f)

print("\n" + "="*45)
print("  RÉSUMÉ ENCODAGE")
print("="*45)
for name, s in splits_encoded.items():
    print(f"\n  {name} ({s['task']}) :")
    print(f"    Xtr : {s['Xtr'].shape}")
    print(f"    Xte : {s['Xte'].shape}")
    print(f"    Features : {s['feats']}")

print("\n✅ splits_encoded.pkl sauvegardé")
print("✅ label_encoders.pkl sauvegardé")
print("✅ label_mappings.pkl sauvegardé")


  RÉSUMÉ ENCODAGE

  Type_Of_Failure (clf) :
    Xtr : (14658, 6)
    Xte : (3665, 6)
    Features : ['Hour', 'Reaction Time (min)', 'Machine_enc', 'Defect Code_enc', 'Shift_enc', 'Equipe_enc']

  Serial_Device (clf) :
    Xtr : (14658, 7)
    Xte : (3665, 7)
    Features : ['Hour', 'Reaction Time (min)', 'Machine_enc', 'Type Of Failure_enc', 'Defect Code_enc', 'Shift_enc', 'Equipe_enc']

  Down_Time (reg) :
    Xtr : (14658, 9)
    Xte : (3665, 9)
    Features : ['Hour', 'Reaction Time (min)', 'Waiting Time (min)', 'Machine_enc', 'Type Of Failure_enc', 'Serial Device_enc', 'Defect Code_enc', 'Shift_enc', 'Equipe_enc']

  When_Panne (reg) :
    Xtr : (14558, 13)
    Xte : (3640, 13)
    Features : ['Last_Hour', 'Last_Down_Time', 'Last_Reaction_Time', 'Last_Waiting_Time', 'Nb_Pannes_Avant', 'Mean_Down_Time', 'Mean_Time_Between', 'Machine_enc', 'Equipe_enc', 'Last_Type_Failure_enc', 'Last_Defect_Code_enc', 'Last_Serial_Device_enc', 'Last_Shift_enc']

✅ splits_encoded.pkl sauvegardé
✅ la